[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S07/S07_05_knn_naive_bayes.ipynb)

# S07 · 05 — k vecinos (k-NN) y Naive Bayes

**Tiempo estimado:** 55 min · **Sprint 7** · M3 (UD 3.5)

**Contexto TurisData.** Tenemos ya la regresión logística como referencia para predecir
cancelaciones. Probamos dos algoritmos con una filosofía muy distinta:
- **k-NN:** «dime con quién se parece esta reserva y te digo qué hará» (razona por **analogía**).
- **Naive Bayes:** «combino la probabilidad de cada dato por separado» (razona con **probabilidades**).

**Al terminar sabrás:**
1. Explicar cómo predice k-NN y por qué **necesita escalar** las variables.
2. Elegir `k` con validación cruzada y entender su efecto (sesgo–varianza).
3. Explicar la idea de Naive Bayes y su suposición «ingenua» de independencia.
4. Comparar los cuatro modelos (logística, k-NN, dos Naive Bayes) por métrica, velocidad y explicabilidad.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, KBinsDiscretizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)


def prep_onehot():
    """Imputar + escalar (numéricas) e imputar + one-hot denso (categóricas)."""
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
    ])

## 1. k-NN: predecir mirando a los vecinos

**Idea.** Para clasificar una reserva nueva:
1. calculamos su **distancia** a todas las reservas conocidas,
2. nos quedamos con las **k más cercanas** (sus «vecinas»),
3. la clase predicha es la **más frecuente entre ellas** (y la probabilidad, la proporción).

No hay «entrenamiento» real: el modelo **guarda los datos** y trabaja al predecir (por eso
puede ser lento con muchos datos). Y como todo depende de **distancias**, una variable con
valores grandes (p. ej. `antelacion_dias`, de 0 a 400) aplasta a las pequeñas
(`cliente_repetidor`, 0 o 1). **Escalar es obligatorio.**

In [ ]:
# ¿Qué variable manda en la distancia? Comparamos qué % de la distancia² entre pares aporta cada una
rng = np.random.default_rng(SEMILLA)
numericas = df[columnas_num].copy()
def reparto_distancia(tabla, n_pares=3000):
    i = rng.integers(0, len(tabla), n_pares)
    j = rng.integers(0, len(tabla), n_pares)
    cuadrados = (tabla.iloc[i].to_numpy() - tabla.iloc[j].to_numpy()) ** 2
    return pd.Series(cuadrados.mean(axis=0) / cuadrados.mean(axis=0).sum(), index=tabla.columns)

sin_escalar = reparto_distancia(numericas)
escalado = reparto_distancia(pd.DataFrame(StandardScaler().fit_transform(numericas), columns=columnas_num))
pd.DataFrame({"sin escalar": sin_escalar, "escaladas": escalado}).mul(100).round(1)

**Ejercicio 1.** Guarda en `pct_antelacion_sin_escalar` y `pct_antelacion_escalada` el % de la
distancia que aporta `antelacion_dias` (de las Series `sin_escalar` y `escalado`, en tanto por
ciento). ¿Qué pasa con las demás variables cuando no se escala?

In [ ]:
# TODO: sin_escalar["antelacion_dias"] * 100
...
assert pct_antelacion_sin_escalar > 60, "Sin escalar, la antelación debería dominar la distancia (más del 60 %)"
assert pct_antelacion_escalada < 25, "Escaladas, cada variable debería aportar una parte parecida"
print(f"Sin escalar, la antelación decide el {pct_antelacion_sin_escalar:.0f} % de la distancia; escalada, el {pct_antelacion_escalada:.0f} %.")

## 2. Elegir `k`

- `k` **pequeño** (1, 3): el modelo copia el ruido de los vecinos (**alta varianza**, sobreajuste).
- `k` **grande**: promedia demasiado y suaviza en exceso (**alto sesgo**).

Se elige con validación cruzada. Usamos el **AUC** para evaluar.

**Ejercicio 2.** Para cada `k` de la lista `ks`, calcula el AUC medio en validación cruzada
(5 folds, `scoring="roc_auc"`) de un `Pipeline` con `prep_onehot()` y `KNeighborsClassifier(n_neighbors=k)`.
Guarda las medias en la lista `auc_por_k` y el mejor valor de `k` en `mejor_k`.

In [ ]:
ks = [1, 5, 15, 31, 61, 101, 201, 401]
# TODO: un for sobre ks con cross_val_score; ks[int(np.argmax(auc_por_k))] da el mejor k
...
assert len(auc_por_k) == len(ks), "Una media por cada k"
assert mejor_k >= 31, "Con estos datos ruidosos, los k grandes funcionan mejor (≥ 31)"
assert auc_por_k[0] < max(auc_por_k) - 0.05, "k = 1 debería ser claramente peor (sobreajuste)"

plt.figure(figsize=(7, 3.8))
plt.plot(ks, auc_por_k, marker="o", color="#2a6f97")
plt.xscale("log")
plt.axvline(mejor_k, color="#e63946", ls="--", label=f"Mejor k = {mejor_k}")
plt.title("k-NN: AUC según el número de vecinos")
plt.xlabel("k (escala logarítmica)")
plt.ylabel("AUC medio en validación cruzada")
plt.legend()
plt.show()

## 3. Naive Bayes: probabilidades multiplicadas

**Idea.** El teorema de Bayes (ya lo has usado en el Sprint 5) permite calcular
`P(cancela | datos de la reserva)`. Naive Bayes hace una simplificación «ingenua»:
**supone que las variables son independientes entre sí dado el resultado**, así que basta
**multiplicar** la probabilidad que cada dato aporta por separado:

`P(cancela | datos) ∝ P(cancela) · P(antelación | cancela) · P(tarifa | cancela) · …`

Es falso (la antelación y el precio pueden estar relacionados), pero **funciona sorprendentemente
bien**, entrena en milisegundos y necesita pocos datos. Sus probabilidades, en cambio, suelen
estar mal calibradas (demasiado extremas). Variantes según el tipo de dato:
- **`GaussianNB`:** variables numéricas (supone una campana de Gauss por clase).
- **`CategoricalNB`:** variables categóricas (cuenta frecuencias). Con numéricas, se **discretizan** en tramos.

**Ejercicio 3.** Crea `nb_gauss`, un `Pipeline` con `prep_onehot()` y `GaussianNB()`. Guarda en
`auc_nb_gauss` su AUC medio en validación cruzada (mismo `cv`).

In [ ]:
# TODO: Pipeline([("prep", prep_onehot()), ("nb", GaussianNB())]) y cross_val_score(..., scoring="roc_auc")
...
assert 0.65 < auc_nb_gauss < 0.85, "El AUC de GaussianNB debería estar entre 0,65 y 0,85"

**Ejercicio 4.** `CategoricalNB` necesita números enteros. Crea `nb_cat`, un `Pipeline` con:
- las numéricas → imputar mediana y `KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")`,
- las categóricas → imputar moda y `OrdinalEncoder()`,
- `CategoricalNB(min_categories=5)`.

Guarda en `auc_nb_cat` su AUC medio en validación cruzada.

In [ ]:
# TODO: ColumnTransformer con dos ramas y CategoricalNB al final
...
assert 0.65 < auc_nb_cat < 0.85, "El AUC de CategoricalNB debería estar entre 0,65 y 0,85"
print(f"AUC GaussianNB: {auc_nb_gauss:.3f} · CategoricalNB: {auc_nb_cat:.3f}")

## 4. Mini-reto: ¿qué modelo para el equipo de reservas?

Crea la tabla `tabla` con una fila por modelo (`"logistica"`, `"knn"`, `"nb_gauss"`, `"nb_cat"`) y
las columnas:
- `auc_cv`: AUC medio (validación cruzada),
- `segundos_ajuste`: tiempo en ajustar con todo `X_train` (`time.perf_counter()`),
- `segundos_predecir`: tiempo en predecir las probabilidades de `X_test`.

Usa `LogisticRegression(max_iter=1000)` y el mejor `k` de antes. Después, en `recomendacion`,
escribe tu elección en 2 frases razonando con las tres columnas y con la **explicabilidad**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
candidatos = {
    "logistica": Pipeline([("prep", prep_onehot()), ("clf", LogisticRegression(max_iter=1000))]),
    "knn": Pipeline([("prep", prep_onehot()), ("clf", KNeighborsClassifier(n_neighbors=mejor_k))]),
    "nb_gauss": nb_gauss,
    "nb_cat": nb_cat,
}
# TODO: por cada candidato: cross_val_score; luego mide el tiempo de fit y de predict_proba
...
assert list(tabla.index) == ["logistica", "knn", "nb_gauss", "nb_cat"], "Filas en el orden pedido"
assert {"auc_cv", "segundos_ajuste", "segundos_predecir"} <= set(tabla.columns)
assert tabla["auc_cv"].between(0.6, 0.9).all(), "AUC entre 0,6 y 0,9 en todos"
assert isinstance(recomendacion, str) and len(recomendacion) > 40, "Escribe una recomendación razonada"
tabla

Observa que **k-NN tarda más en predecir que en «ajustar»** (al revés que casi todos los demás):
tiene que comparar cada reserva nueva con todas las guardadas. En producción, con millones de
reservas, eso pesa.

## 5. Para reflexionar

1. ¿Por qué `k = 1` funciona tan mal aquí? ¿Qué pasaría con datos casi sin ruido?
2. Naive Bayes «se equivoca» al suponer independencia y aun así funciona. ¿Cuándo te fiarías menos de él?
3. Un cliente pregunta «¿por qué el modelo cree que esta reserva se cancelará?». ¿Con cuál de los cuatro modelos puedes responder mejor?
4. ¿Cómo cambia la respuesta si el modelo debe dar 1000 predicciones por segundo?

## 6. Resumen: qué has aprendido
- **k-NN** predice por analogía; exige escalar y `k` se elige con validación cruzada; es lento al predecir.
- **Naive Bayes** multiplica probabilidades asumiendo independencia; es rápido y sencillo.
- Siempre se comparan **métrica + coste + explicabilidad**: el «mejor» modelo depende del contexto.